install libraries

In [0]:
%pip install requests azure-storage-blob

Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.0/407.0 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 20.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 6.7 MB/s eta 0:00:00
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.4.0
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.10/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-c0bf1888-8bc4-4aad-a8e9-7a9efd289a9f
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
Note: you may need to restart the kernel using dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


setup auth

In [0]:
API_URL = "https://api.restful-api.dev/objects"
AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=fhir;AccountKey=TSeW7zSropN7/KO6Hd1NxUzfFXII93rRCTBGaY27B1+dOLeaSBVQ+leiR4Vbv03ANLv3KGkwGtnS+AStcGFPRg==;EndpointSuffix=core.windows.net"
CONTAINER_NAME = "api-container"
BLOB_PREFIX = "api_data"
HEADERS = {
    "Accept": "application/json",
    # "Authorization": "Bearer YOUR_TOKEN"
}


call API

In [0]:
import requests
import json

response = requests.get(API_URL, headers=HEADERS)
response.raise_for_status()
data = response.json()
print("Sample data:", json.dumps(data, indent=2)[:1000])


Sample data: [
  {
    "id": "1",
    "name": "Google Pixel 6 Pro",
    "data": {
      "color": "Cloudy White",
      "capacity": "128 GB"
    }
  },
  {
    "id": "2",
    "name": "Apple iPhone 12 Mini, 256GB, Blue",
    "data": null
  },
  {
    "id": "3",
    "name": "Apple iPhone 12 Pro Max",
    "data": {
      "color": "Cloudy White",
      "capacity GB": 512
    }
  },
  {
    "id": "4",
    "name": "Apple iPhone 11, 64GB",
    "data": {
      "price": 389.99,
      "color": "Purple"
    }
  },
  {
    "id": "5",
    "name": "Samsung Galaxy Z Fold2",
    "data": {
      "price": 689.99,
      "color": "Brown"
    }
  },
  {
    "id": "6",
    "name": "Apple AirPods",
    "data": {
      "generation": "3rd",
      "price": 120
    }
  },
  {
    "id": "7",
    "name": "Apple MacBook Pro 16",
    "data": {
      "year": 2019,
      "price": 1849.99,
      "CPU model": "Intel Core i9",
      "Hard disk size": "1 TB"
    }
  },
  {
    "id": "8",
    "name": "Apple Watch Series 8",

upload to blob

In [0]:
from azure.storage.blob import BlobServiceClient
from datetime import datetime
import io

blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)

timestamp = datetime.utcnow().strftime("%Y%m%d%H%M%S")
blob_name = f"raw/{BLOB_PREFIX}_{timestamp}.json"

json_string = json.dumps(data, indent=2)
blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=blob_name)
blob_client.upload_blob(io.BytesIO(json_string.encode()), overwrite=True)

print(f"Uploaded to Azure Blob Storage: {CONTAINER_NAME}/{blob_name}")


Uploaded to Azure Blob Storage: api-container/raw/api_data_20250713153031.json
